# TUGAS 4.2: TF-IDF (Pembobotan Term)
**Sistem Temu Kembali Informasi**

* **Nama    :** Washiatul Akmal
* **NIM     :** 240210501050
* **Kelas   :** Sistem Temu Kembali | Tekom Pilihan B 24

In [ ]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Empat Dokumen Singkat (1-3 Kalimat) dengan Irisan Kata
dokumen_mentah = [
    "Sistem komputer dan jaringan komputer saling terhubung.",
    "Jaringan komputer menghubungkan sistem informasi kampus.",
    "Sistem temu kembali informasi memanfaatkan kecerdasan buatan.",
    "Kecerdasan buatan mendukung sistem komputer cerdas."
]

# Stopwords sederhana Bahasa Indonesia
stop_words = {"dan", "saling", "memanfaatkan", "mendukung", "menghubungkan"}

dokumen_bersih = []
tokens_korpus = []

for text in dokumen_mentah:
    text_clean = re.sub(r'[^a-z\s]', '', text.lower())
    tokens = [w for w in text_clean.split() if w not in stop_words]
    tokens_korpus.append(tokens)
    dokumen_bersih.append(" ".join(tokens))

# Tampilkan hasil preprocessing sederhana
print("=== HASIL PREPROCESSING SEDERHANA ===")
for i, (asli, bersih) in enumerate(zip(dokumen_mentah, dokumen_bersih), 1):
    print(f"D{i} Asli   : {asli}")
    print(f"D{i} Bersih : {bersih}\n")

# Pembentukan Vokabulari Kata Unik
vokabulari = sorted(list(set([kata for sublist in tokens_korpus for kata in sublist])))
print(f"Vokabulari ({len(vokabulari)} term unik):", vokabulari)

In [ ]:
# Matriks Bag-of-Words (Raw Count)
bow_dict = {}
for term in vokabulari:
    bow_dict[term] = [doc.count(term) for doc in tokens_korpus]

df_bow = pd.DataFrame(bow_dict, index=["D1", "D2", "D3", "D4"]).T
print("=== MATRIKS BAG-OF-WORDS (RAW COUNT) ===")
display(df_bow)

### Rumus Perhitungan Manual
1. **Term Frequency (Raw Count):** $\text{TF}(t, d) = f(t, d)$[cite: 6]
2. **Document Frequency:** $\text{DF}(t) = \text{jumlah dokumen yang memuat kata } t$[cite: 6]
3. **Inverse Document Frequency:** $\text{IDF}(t) = \log_{10}\left(\frac{N}{\text{DF}(t)}\right)$ dengan $N = 4$[cite: 6]
4. **Bobot TF-IDF:** $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$[cite: 6, 7]

In [ ]:
N_dok = 4

# 1. Tabel TF (Raw Count)
df_tf = df_bow.copy()

# 2 & 3. Tabel DF dan IDF Manual
df_count = (df_bow > 0).sum(axis=1)
idf_manual = df_count.apply(lambda df_val: math.log10(N_dok / df_val))

df_df_idf = pd.DataFrame({
    "DF": df_count,
    "N / DF": [f"{N_dok}/{val}" for val in df_count],
    "Rumus IDF": [f"log10({N_dok}/{val})" for val in df_count],
    "Nilai IDF": idf_manual.round(4)
})

# 4. Matriks TF-IDF Manual
df_tfidf_manual = df_tf.copy().astype(float)
for term in vokabulari:
    df_tfidf_manual.loc[term] = df_tfidf_manual.loc[term] * idf_manual[term]

print("=== TABEL TERM FREQUENCY (TF) ===")
display(df_tf)

print("\n=== TABEL DF & IDF (MANUAL log10) ===")
display(df_df_idf)

print("\n=== MATRIKS TF-IDF MANUAL (TF x IDF) ===")
display(df_tfidf_manual.round(4))

In [ ]:
vectorizer = TfidfVectorizer()
matriks_sklearn = vectorizer.fit_transform(dokumen_bersih)

df_sklearn = pd.DataFrame(
    matriks_sklearn.toarray(),
    index=["D1", "D2", "D3", "D4"],
    columns=vectorizer.get_feature_names_out()
).T

print("=== MATRIKS TF-IDF HASIL SCIKIT-LEARN ===")
display(df_sklearn.round(4))

### Perbandingan Nilai Manual vs Scikit-Learn

Terdapat perbedaan angka antara kalkulasi manual dengan keluaran `TfidfVectorizer` Scikit-Learn:
* **Formula IDF:** Perhitungan manual menggunakan rumus klasik $\log_{10}(N / \text{DF})$[cite: 6]. Scikit-Learn menggunakan logaritma natural ($\ln$) dengan *smoothing* bawaan: $\text{IDF}(t) = \ln\left(\frac{1 + N}{1 + \text{DF}}\right) + 1$ guna mencegah galat nilai nol.
* **Kata Bernilai Nol:** Kata `"sistem"` pada manual memperoleh bobot `0.0000` karena $\log_{10}(4/4) = 0$. Di Scikit-Learn, nilai kata tersebut tetap positif.
* **Normalisasi L2:** Scikit-Learn secara default menerapkan normalisasi Euclidean ($L_2$) pada setiap baris vektor dokumen sehingga total kuadrat bobot bernilai 1 ($\sum w_i^2 = 1$)[cite: 6], sedangkan hitungan manual menggunakan perkalian langsung tanpa normalisasi.